# Teorema do Limite Central — uma simulação

**Objetivo:** visualizar por que a média amostral tende a ter distribuição aproximadamente Normal, mesmo quando a variável original não é Normal.

Em Econometria, essa ideia aparece o tempo todo. Em particular, ela ajuda a entender por que muitos estimadores têm distribuições aproximadamente normais em amostras grandes.

---

## O que diz o TLC?

Considere variáveis aleatórias independentes e identicamente distribuídas:

\[
X_1, X_2,\ldots,X_n
\]

com média populacional \(\mu\) e variância finita \(\sigma^2\).

A média amostral é

\[
\bar X = \frac{1}{n}\sum_{i=1}^{n}X_i.
\]

Para \(n\) suficientemente grande,

\[
\bar X \approx N\left(\mu,\frac{\sigma^2}{n}\right).
\]

Equivalentemente,

\[
\frac{\bar X-\mu}{\sigma/\sqrt{n}} \approx N(0,1).
\]

**Duas mensagens importantes:**

1. Não é necessário que \(X\) tenha distribuição Normal.
2. À medida que \(n\) aumenta, a distribuição de \(\bar X\) fica mais concentrada em torno de \(\mu\).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

np.random.seed(123)

## 1. Começamos com uma população que NÃO é Normal

Vamos usar uma distribuição exponencial:

\[
X \sim \text{Exponencial}(1).
\]

Ela é fortemente assimétrica à direita.

Para essa distribuição:

\[
E(X)=1
\]

e

\[
Var(X)=1.
\]

In [ ]:
pop = np.random.exponential(scale=1, size=200_000)

plt.figure(figsize=(9, 4.5))
plt.hist(pop, bins=80, density=True, alpha=0.75)
plt.axvline(pop.mean(), linestyle="--", linewidth=2, label=f"Média ≈ {pop.mean():.2f}")
plt.xlim(0, 8)
plt.xlabel("X")
plt.ylabel("Densidade")
plt.title("População: distribuição exponencial")
plt.legend()
plt.show()

print(f"Média simulada: {pop.mean():.4f}")
print(f"Desvio-padrão simulado: {pop.std():.4f}")

### Pergunta para a turma

A variável \(X\) parece ter distribuição Normal?

**Não.** Ela é bastante assimétrica.

Agora vamos perguntar algo diferente:

> O que acontece com a distribuição da **média de várias observações de \(X\)**?

## 2. Repetindo o experimento de amostragem

Imagine o seguinte experimento:

1. retiramos uma amostra de tamanho \(n\);
2. calculamos a média \(\bar X\);
3. repetimos isso muitas vezes.

Ao final teremos milhares de valores de \(\bar X\).

Essa coleção de médias forma a **distribuição amostral da média**.

In [ ]:
def simula_medias(n, R=20_000):
    # Gera R amostras de tamanho n de uma Exponencial(1)
    # e retorna as R médias amostrais.
    amostras = np.random.exponential(scale=1, size=(R, n))
    return amostras.mean(axis=1)

## 3. O efeito do tamanho da amostra

Vamos comparar:

- \(n=1\)
- \(n=5\)
- \(n=30\)
- \(n=100\)

Em cada caso serão simuladas **20.000 médias amostrais**.

In [ ]:
ns = [1, 5, 30, 100]

for n in ns:
    medias = simula_medias(n)

    mu = 1
    sigma = 1
    erro_padrao = sigma / np.sqrt(n)

    x = np.linspace(
        max(0, mu - 4 * erro_padrao),
        mu + 5 * erro_padrao,
        500
    )

    plt.figure(figsize=(9, 4.5))
    plt.hist(
        medias, bins=60, density=True, alpha=0.70,
        label="Distribuição simulada de $\\bar X$"
    )
    plt.plot(
        x,
        norm.pdf(x, loc=mu, scale=erro_padrao),
        linewidth=2.5,
        label="Normal prevista pelo TLC"
    )
    plt.axvline(mu, linestyle="--", linewidth=2, label=r"$\mu=1$")

    plt.xlabel(r"$\bar X$")
    plt.ylabel("Densidade")
    plt.title(f"Distribuição da média amostral — n = {n}")
    plt.legend()
    plt.show()

    print(f"n = {n}")
    print(f"  Média das médias:              {medias.mean():.4f}")
    print(f"  DP das médias (simulado):      {medias.std():.4f}")
    print(f"  Erro-padrão teórico 1/sqrt(n): {erro_padrao:.4f}")
    print("-" * 55)

## 4. O que deveríamos observar?

Quando \(n\) aumenta:

### A média não muda

\[
E(\bar X)=\mu.
\]

Neste exemplo,

\[
E(\bar X)=1.
\]

### A dispersão diminui

\[
SD(\bar X)=\frac{\sigma}{\sqrt n}.
\]

Como \(\sigma=1\),

\[
SD(\bar X)=\frac{1}{\sqrt n}.
\]

| \(n\) | \(SD(\bar X)\) |
|---:|---:|
| 1 | 1.000 |
| 5 | 0.447 |
| 30 | 0.183 |
| 100 | 0.100 |

### E a forma fica cada vez mais próxima de uma Normal

Mesmo que a população original seja bastante assimétrica.

## 5. Vendo os quatro casos juntos

Esta figura deixa especialmente claro que há **duas coisas acontecendo ao mesmo tempo**:

1. a distribuição vai adquirindo formato aproximadamente Normal;
2. ela vai ficando mais estreita porque o erro-padrão diminui com \(n\).

In [ ]:
medias_por_n = {n: simula_medias(n) for n in ns}

plt.figure(figsize=(10, 5))

for n in ns:
    medias = medias_por_n[n]
    plt.hist(
        medias,
        bins=80,
        density=True,
        histtype="step",
        linewidth=2,
        label=f"n={n}"
    )

plt.axvline(1, linestyle="--", linewidth=2)
plt.xlim(0, 4)
plt.xlabel(r"$\bar X$")
plt.ylabel("Densidade")
plt.title("Distribuição amostral da média para diferentes tamanhos de amostra")
plt.legend()
plt.show()

## 6. A versão padronizada do TLC

O TLC também pode ser escrito como

\[
Z=
\frac{\bar X-\mu}{\sigma/\sqrt n}
\approx N(0,1).
\]

Vamos verificar isso empiricamente.

In [ ]:
R = 30_000
n = 30

medias = simula_medias(n, R)

mu = 1
sigma = 1

z = (medias - mu) / (sigma / np.sqrt(n))

x = np.linspace(-4, 4, 500)

plt.figure(figsize=(9, 4.5))
plt.hist(z, bins=70, density=True, alpha=0.70,
         label="Valores padronizados simulados")
plt.plot(x, norm.pdf(x), linewidth=2.5,
         label="Normal padrão N(0,1)")
plt.xlabel("z")
plt.ylabel("Densidade")
plt.title("TLC: média amostral padronizada")
plt.legend()
plt.show()

print(f"Média de Z: {z.mean():.4f}")
print(f"Desvio-padrão de Z: {z.std():.4f}")

## 7. Por que isso é importante para Econometria?

Em Econometria frequentemente trabalhamos com estimadores que podem ser escritos, exatamente ou aproximadamente, como **médias de muitas contribuições individuais**.

Por exemplo,

\[
\bar Y=\frac{1}{n}\sum_{i=1}^n Y_i.
\]

Sob condições adequadas, o TLC implica que

\[
\sqrt n(\bar Y-\mu)
\]

tem distribuição aproximadamente Normal.

A mesma lógica está por trás da **normalidade assintótica** de muitos estimadores econométricos, inclusive o estimador de MQO sob condições apropriadas.

Isso permite construir:

- erros-padrão;
- intervalos de confiança;
- testes \(t\);
- testes de hipóteses em geral.

A Normal que aparece na Econometria muitas vezes **não vem da hipótese de que os dados sejam normais**. Ela aparece como consequência de resultados assintóticos como o TLC.

# 8. Experimento para fazer em sala

Altere apenas o valor de `n` abaixo.

Sugestões: `2`, `5`, `10`, `30`, `100`, `1000`.

Observe:

1. a forma do histograma;
2. a média das médias;
3. o desvio-padrão das médias;
4. a aproximação pela Normal.

In [ ]:
# ALTERE AQUI
n = 10

R = 20_000
medias = simula_medias(n, R)

mu = 1
sigma = 1
erro_padrao = sigma / np.sqrt(n)

x = np.linspace(
    max(0, mu - 4 * erro_padrao),
    mu + 5 * erro_padrao,
    500
)

plt.figure(figsize=(9, 4.5))
plt.hist(medias, bins=60, density=True, alpha=0.70,
         label="Simulação")
plt.plot(
    x,
    norm.pdf(x, loc=mu, scale=erro_padrao),
    linewidth=2.5,
    label="Normal do TLC"
)
plt.axvline(mu, linestyle="--", linewidth=2)

plt.xlabel(r"$\bar X$")
plt.ylabel("Densidade")
plt.title(f"Experimento — n = {n}")
plt.legend()
plt.show()

print(f"Média teórica de X̄:        {mu:.4f}")
print(f"Média simulada de X̄:      {medias.mean():.4f}")
print(f"EP teórico de X̄:          {erro_padrao:.4f}")
print(f"DP simulado de X̄:         {medias.std():.4f}")

# 9. Perguntas para discussão

1. Para \(n=1\), qual é a relação entre a distribuição de \(X\) e a distribuição de \(\bar X\)?
2. O valor esperado de \(\bar X\) muda quando aumentamos \(n\)?
3. Por que a distribuição das médias fica mais estreita?
4. Para qual valor de \(n\) a aproximação Normal parece razoável neste exemplo?
5. O TLC afirma que os **dados** tornam-se normais quando \(n\) aumenta?
6. Qual objeto se torna aproximadamente Normal?
7. Por que isso é útil para inferência estatística e Econometria?

---

## Mensagem principal

> **O Teorema do Limite Central não diz que os dados se tornam normais. Ele diz que, sob condições adequadas, a distribuição de uma média (ou soma padronizada) torna-se aproximadamente Normal quando o tamanho da amostra cresce.**